In [ ]:
import time, random, re, json, hashlib
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
NOTEBOOK_START = time.time()
elapsed = lambda: time.time() - NOTEBOOK_START
IN_KAGGLE = Path('/kaggle').exists()

def hms(s):
    s = max(0, int(s))
    return f'{s // 3600:02d}:{s % 3600 // 60:02d}:{s % 60:02d}'


In [ ]:
CANDIDATE_DATA_ROOTS = [Path('/kaggle/input/datasets/xdxd003'), Path('/kaggle/input')]

def find_data_root():
    for root in CANDIDATE_DATA_ROOTS:
        if not root.exists():
            continue
        if (root / 'original').exists():
            return root
        for p in root.glob('**/original'):
            return p.parent
    return CANDIDATE_DATA_ROOTS[0]
DATA_ROOT = find_data_root()
BASE_ROOT = (Path('/kaggle/working') if IN_KAGGLE else Path('./working')) / 'FAKED_STUDENT'
WORK_ROOT = BASE_ROOT / 'initializing'
FACE_ROOT, MASK_ROOT = (WORK_ROOT / 'faces', WORK_ROOT / 'masks')
for d in (FACE_ROOT / 'real', FACE_ROOT / 'fake', MASK_ROOT):
    d.mkdir(parents=True, exist_ok=True)
REAL_METHOD = 'original'
FAKE_METHODS_KNOWN = ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures', 'FaceShifter', 'DeepFakeDetection']
N_SOURCE_VIDEOS = 240
FRAMES_PER_VIDEO = 8
MAX_FRAME_SCAN = 200
IMAGE_SIZE = 224
FACE_PADDING = 0.25
DETECTION_WIDTH = 480
JPEG_QUALITY = 95
MASK_MIN_DIFF = 14
MASK_MIN_AREA = 0.004
MASK_MAX_AREA = 0.65
TRAIN_FRAC, VAL_FRAC = (0.7, 0.15)
NUM_WORKERS = 2 if IN_KAGGLE else 0
PREPROCESS_TIME_BUDGET_SEC = 90 * 60
TOTAL_TIME_BUDGET_SEC = PREPROCESS_TIME_BUDGET_SEC


In [ ]:
assert DATA_ROOT.exists(), f'Dataset root not found: {DATA_ROOT}'
video_paths = sorted(DATA_ROOT.rglob('*.mp4'))
assert video_paths, f'No MP4 videos found under {DATA_ROOT}'
records = []
for vp in video_paths:
    method = vp.parent.name
    if method != REAL_METHOD and method not in FAKE_METHODS_KNOWN:
        continue
    ids = re.findall('\\d+', vp.stem)
    records.append({'video_id': vp.stem, 'video_path': str(vp), 'label': 0 if method == REAL_METHOD else 1, 'class_name': 'real' if method == REAL_METHOD else 'fake', 'manipulation_method': method, 'primary_source_id': ids[0] if ids else vp.stem})
video_manifest = pd.DataFrame(records).drop_duplicates('video_path').reset_index(drop=True)
real_manifest = video_manifest[video_manifest.class_name == 'real'].reset_index(drop=True)
fake_manifest = video_manifest[video_manifest.class_name == 'fake'].reset_index(drop=True)
methods_available = sorted(fake_manifest.manipulation_method.unique())
print(f'Videos indexed: {len(video_manifest):,}  (real {len(real_manifest):,} | fake {len(fake_manifest):,})')
print(f'Fake methods  : {methods_available}')
fake_lookup = defaultdict(dict)
for _, row in fake_manifest.iterrows():
    fake_lookup[row['primary_source_id']][row['manipulation_method']] = row
pairs, mi = ([], 0)
for _, r in real_manifest.sample(frac=1.0, random_state=SEED).iterrows():
    candidates = fake_lookup.get(r['primary_source_id'])
    if not candidates:
        continue
    chosen = None
    for _ in range(len(methods_available)):
        m = methods_available[mi % len(methods_available)]
        mi += 1
        if m in candidates:
            chosen = candidates[m]
            break
    chosen = chosen if chosen is not None else next(iter(candidates.values()))
    pairs.append({'video_id': r['video_id'], 'real_row': r, 'fake_row': chosen})
    if len(pairs) >= N_SOURCE_VIDEOS:
        break
assert pairs, 'No real/fake pairs formed — check source-id parsing for this dataset mirror.'
print(f'\nPaired {len(pairs)} videos (requested {N_SOURCE_VIDEOS}).')
print(pd.Series([p['fake_row']['manipulation_method'] for p in pairs]).value_counts().to_string())
import hashlib
SETTINGS = {'selected_paths': [(str(x['real_row']['video_path']), str(x['fake_row']['video_path'])) for x in pairs], 'seed': SEED, 'source_pairs': N_SOURCE_VIDEOS, 'frames_per_video': FRAMES_PER_VIDEO, 'max_frame_scan': MAX_FRAME_SCAN, 'resolution': IMAGE_SIZE, 'face_padding': FACE_PADDING, 'detection_width': DETECTION_WIDTH, 'jpeg_quality': JPEG_QUALITY, 'mask_min_diff': MASK_MIN_DIFF, 'mask_min_area': MASK_MIN_AREA, 'mask_max_area': MASK_MAX_AREA, 'split': (TRAIN_FRAC, VAL_FRAC), 'preprocessing_revision': 'standalone_v1'}
RUN_ID = hashlib.sha256(json.dumps(SETTINGS, sort_keys=True).encode()).hexdigest()[:12]
WORK_ROOT = BASE_ROOT / 'experiments' / RUN_ID
FACE_ROOT, MASK_ROOT = (WORK_ROOT / 'faces', WORK_ROOT / 'masks')
REPORT_DIR = WORK_ROOT / 'report_artifacts'
for d in [FACE_ROOT / 'real', FACE_ROOT / 'fake', MASK_ROOT, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
(REPORT_DIR / 'run_settings.json').write_text(json.dumps({'run_id': RUN_ID, **SETTINGS}, indent=2))
video_manifest.to_csv(REPORT_DIR / 'all_indexed_videos.csv', index=False)
PLOT_DIR = REPORT_DIR
pd.DataFrame([{'real_source_id': str(p['video_id']), 'real_video_path': str(p['real_row']['video_path']), 'fake_video_path': str(p['fake_row']['video_path']), 'manipulation_method': str(p['fake_row']['manipulation_method'])} for p in pairs]).to_csv(REPORT_DIR / 'selected_video_pairs.csv', index=False)
print('Run ID:', RUN_ID, 'Report output:', REPORT_DIR)


In [ ]:
face_cascade = cv2.CascadeClassifier(str(Path(cv2.data.haarcascades) / 'haarcascade_frontalface_default.xml'))
assert not face_cascade.empty(), 'OpenCV Haar cascade failed to load.'

def frame_count_of(path):
    cap = cv2.VideoCapture(str(path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if cap.isOpened() else 0
    cap.release()
    return n

def read_frames_at(path, indices):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        cap.release()
        return {}
    wanted = sorted({int(i) for i in indices})
    out, pos, w = ({}, 0, 0)
    while w < len(wanted):
        if not cap.grab():
            break
        if pos == wanted[w]:
            ok, frame = cap.retrieve()
            if ok:
                out[wanted[w]] = frame
            w += 1
        pos += 1
    cap.release()
    return out

def detect_face(frame):
    h, w = frame.shape[:2]
    scale = min(1.0, DETECTION_WIDTH / w)
    small = cv2.resize(frame, (max(1, int(w * scale)), max(1, int(h * scale))), interpolation=cv2.INTER_AREA) if scale < 1.0 else frame
    gray = cv2.equalizeHist(cv2.cvtColor(small, cv2.COLOR_BGR2GRAY))
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))
    if len(faces) == 0:
        return None
    x, y, fw, fh = max(faces, key=lambda b: b[2] * b[3])
    inv = 1.0 / scale
    return (int(x * inv), int(y * inv), int(fw * inv), int(fh * inv))

def padded_bounds(shape, box, padding=FACE_PADDING):
    x, y, w, h = box
    fh, fw = shape[:2]
    px, py = (int(w * padding), int(h * padding))
    return (max(0, x - px), max(0, y - py), min(fw, x + w + px), min(fh, y + h + py))

def make_pseudo_mask(real_crop, fake_crop):
    gray = cv2.cvtColor(cv2.absdiff(real_crop, fake_crop), cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    otsu_t, _ = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = (gray >= max(MASK_MIN_DIFF, otsu_t)).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8), iterations=2)
    n, lbl, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n <= 1:
        return None
    areas = stats[1:, cv2.CC_STAT_AREA]
    biggest = areas.max()
    keep = [i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= 0.3 * biggest]
    mask = np.isin(lbl, keep).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    frac = float((mask > 0).mean())
    if not MASK_MIN_AREA <= frac <= MASK_MAX_AREA:
        return None
    return mask
CACHE_MANIFEST = WORK_ROOT / 'frame_manifest.csv'
if CACHE_MANIFEST.exists() and (WORK_ROOT / 'preprocess_complete.json').exists():
    frame_manifest = pd.read_csv(CACHE_MANIFEST, dtype={'video_id': str, 'sample_id': str, 'clip_id': str})
    file_check = frame_manifest.image_path.dropna().astype(str).map(lambda p: Path(p).exists())
    if len(frame_manifest) and file_check.all():
        print('Loaded', len(frame_manifest), 'previously preprocessed crops from this exact experiment.')
    else:
        raise RuntimeError('Preprocessing cache is incomplete. Delete the run folder and rerun.')
else:
    t0 = time.time()
    frame_records, stats_counter = ([], defaultdict(int))
    for pair in tqdm(pairs, desc='Preprocessing pairs'):
        if time.time() - t0 > PREPROCESS_TIME_BUDGET_SEC:
            print(f"\nPreprocessing budget reached — stopping with {stats_counter['crops']} crops.")
            break
        real_path = Path(pair['real_row']['video_path'])
        fake_path = Path(pair['fake_row']['video_path'])
        method = pair['fake_row']['manipulation_method']
        n_frames = min(frame_count_of(real_path), frame_count_of(fake_path), MAX_FRAME_SCAN)
        if n_frames <= 0:
            stats_counter['skipped_pairs'] += 1
            continue
        idxs = np.unique(np.linspace(0, n_frames - 1, FRAMES_PER_VIDEO, dtype=int)).tolist()
        real_frames = read_frames_at(real_path, idxs)
        fake_frames = read_frames_at(fake_path, idxs)
        last_box = None
        for i in idxs:
            rf, ff = (real_frames.get(i), fake_frames.get(i))
            if rf is None or ff is None or rf.shape != ff.shape:
                continue
            box = detect_face(rf) or last_box
            if box is None:
                stats_counter['no_face'] += 1
                continue
            last_box = box
            x1, y1, x2, y2 = padded_bounds(rf.shape, box)
            if x2 - x1 < 32 or y2 - y1 < 32:
                continue
            real_crop = cv2.resize(rf[y1:y2, x1:x2], (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
            fake_crop = cv2.resize(ff[y1:y2, x1:x2], (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
            mask = make_pseudo_mask(real_crop, fake_crop)
            sid = f"{pair['video_id']}_{method}_{i:06d}"
            real_out = FACE_ROOT / 'real' / f'{sid}.jpg'
            fake_out = FACE_ROOT / 'fake' / f'{sid}.jpg'
            cv2.imwrite(str(real_out), real_crop, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
            cv2.imwrite(str(fake_out), fake_crop, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
            if mask is not None:
                mask_out = MASK_ROOT / f'{sid}.png'
                cv2.imwrite(str(mask_out), mask)
                stats_counter['valid_masks'] += 1
            else:
                mask_out = None
                stats_counter['rejected_masks'] += 1
            common = {'sample_id': sid, 'video_id': pair['video_id'], 'manipulation_method': method}
            frame_records.append({**common, 'clip_id': f"{pair['video_id']}__real", 'image_path': str(real_out), 'mask_path': None, 'label': 0, 'seg_valid': 0})
            frame_records.append({**common, 'clip_id': f"{pair['video_id']}__fake", 'image_path': str(fake_out), 'mask_path': str(mask_out) if mask_out else None, 'label': 1, 'seg_valid': int(mask is not None)})
            stats_counter['crops'] += 2
    assert stats_counter['crops'] > 0, 'No crops produced — inspect dataset layout / detector.'
    frame_manifest = pd.DataFrame(frame_records)
    frame_manifest.to_csv(CACHE_MANIFEST, index=False)
    (WORK_ROOT / 'preprocess_complete.json').write_text(json.dumps({'run_id': RUN_ID, 'rows': len(frame_manifest)}))
    n_fake = int((frame_manifest.label == 1).sum())
    print(f'\nPreprocessing: {hms(time.time() - t0)}')
    print(f'Crops              : {len(frame_manifest):,} (real {int((frame_manifest.label == 0).sum()):,} | fake {n_fake:,})')
    print(f"Usable seg. masks  : {stats_counter['valid_masks']:,} / {n_fake:,} fakes ({100 * stats_counter['valid_masks'] / max(1, n_fake):.1f}%)")
    print(f"Rejected masks     : {stats_counter['rejected_masks']:,}   detector misses: {stats_counter['no_face']}   skipped pairs: {stats_counter['skipped_pairs']}")
    print(f'Elapsed            : {hms(elapsed())} / {hms(TOTAL_TIME_BUDGET_SEC)}')
raw_count = video_manifest.groupby(['class_name', 'manipulation_method']).size().reset_index(name='n_videos')
raw_count.to_csv(REPORT_DIR / 'available_original_dataset_index.csv', index=False)
print('Actual processed frame crops:', len(frame_manifest), 'valid fake pseudo-masks:', int(frame_manifest.seg_valid.sum()))


In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

class FacedDataset(Dataset):

    def __init__(self, df, train=False, size=IMAGE_SIZE):
        self.df = df.reset_index(drop=True)
        self.train, self.size = (train, size)

    def __len__(self):
        return len(self.df)

    def _augment(self, img, mask):
        if random.random() < 0.5:
            img, mask = (cv2.flip(img, 1), cv2.flip(mask, 1))
        if random.random() < 0.3:
            alpha = random.uniform(0.96, 1.04)
            beta = random.uniform(-5.0, 5.0)
            img = np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
        return (img, mask)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = cv2.imread(row['image_path'])
        if img is None:
            raise FileNotFoundError(f"Missing face crop: {row['image_path']}")
        mp = row['mask_path']
        if isinstance(mp, str) and mp:
            m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
            mask = (m > 127).astype(np.float32) if m is not None else np.zeros((self.size, self.size), np.float32)
        else:
            mask = np.zeros((self.size, self.size), np.float32)
        if self.train:
            img, mask = self._augment(img, mask)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        img = (img - IMAGENET_MEAN) / IMAGENET_STD
        return (torch.from_numpy(img).permute(2, 0, 1).float(), torch.tensor(int(row['label']), dtype=torch.long), torch.from_numpy(mask).unsqueeze(0).float(), torch.tensor(float(row['seg_valid']), dtype=torch.float32))
source_map = {}
for pair in pairs:
    vid = str(pair['video_id'])
    srcs = set(re.findall('\\d+', str(pair['fake_row']['video_id'])))
    srcs.add(vid)
    source_map[vid] = srcs
parent = {v: v for v in source_map}

def _root(v):
    while parent[v] != v:
        parent[v] = parent[parent[v]]
        v = parent[v]
    return v

def _union(a, b):
    a, b = (_root(a), _root(b))
    if a != b:
        parent[b] = a
owner = {}
for vid, src_ids in source_map.items():
    for sid in src_ids:
        if sid in owner:
            _union(vid, owner[sid])
        else:
            owner[sid] = vid
component_of = {v: _root(v) for v in source_map}
vids = sorted({component_of[str(v)] for v in frame_manifest.video_id.unique()})
random.Random(SEED).shuffle(vids)
n_tr = max(1, int(len(vids) * TRAIN_FRAC))
n_va = max(1, int(len(vids) * VAL_FRAC))
split_of = {}
for v in vids[:n_tr]:
    split_of[v] = 'train'
for v in vids[n_tr:n_tr + n_va]:
    split_of[v] = 'val'
for v in vids[n_tr + n_va:]:
    split_of[v] = 'test'
frame_manifest['split'] = frame_manifest['video_id'].astype(str).map(lambda v: split_of[component_of[v]])
assert frame_manifest.split.notna().all()
assert len(vids) >= 3, 'Need at least three source-disjoint groups. Increase N_SOURCE_VIDEOS.'
frame_manifest.to_csv(REPORT_DIR / 'shared_frame_manifest.csv', index=False)
train_df = frame_manifest[frame_manifest.split == 'train'].reset_index(drop=True)
val_df = frame_manifest[frame_manifest.split == 'val'].reset_index(drop=True)
test_df = frame_manifest[frame_manifest.split == 'test'].reset_index(drop=True)
assert len(train_df) and len(val_df) and len(test_df), 'Empty split — raise N_SOURCE_VIDEOS.'
for split_a, split_b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    a = set(frame_manifest[frame_manifest.split == split_a].video_id.astype(str))
    b = set(frame_manifest[frame_manifest.split == split_b].video_id.astype(str))
    assert not a.intersection(b)
    src_a = set().union(*(source_map[v] for v in a)) if a else set()
    src_b = set().union(*(source_map[v] for v in b)) if b else set()
    assert not src_a.intersection(src_b), f'Source leakage between {split_a} and {split_b}'
assert train_df.label.nunique() == 2 and val_df.label.nunique() == 2 and (test_df.label.nunique() == 2)
split_counts = pd.DataFrame([{'split': name, 'source_pairs': int(d.video_id.nunique()), 'real_frames': int((d.label == 0).sum()), 'fake_frames': int((d.label == 1).sum()), 'total_frames': len(d), 'valid_fake_pseudo_masks': int(d.seg_valid.sum())} for name, d in [('train', train_df), ('validation', val_df), ('test', test_df)]])
split_counts.to_csv(REPORT_DIR / 'train_validation_test_counts.csv', index=False)
